In [ ]:
import sys
sys.path.append("..")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from src.scoutai_common import get_engine, load_master_view, engineer_features, route_predict

engine = get_engine()
df = load_master_view(engine)
df = engineer_features(df)
df = route_predict(df)
df['residuals'] = df['current_market_value'] - df['predicted_value']

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
model_colors = {'full': '#e74c3c', 'performance_only': '#9b59b6'}
hist_colors = {'full': '#3498db', 'performance_only': '#2ecc71'}

for row, label in enumerate(['full', 'performance_only']):
    subset = df[df['model_used'] == label]
    sns.scatterplot(x=subset['predicted_value'] / 1e6, y=subset['residuals'] / 1e6,
                     alpha=0.5, color=model_colors[label], edgecolor='k', ax=axes[row, 0])
    axes[row, 0].axhline(y=0, color='black', linestyle='--', linewidth=2)
    axes[row, 0].set_title(f"Residuals vs. Predicted -- {label} model (n={len(subset)})", fontsize=13, fontweight='bold')
    axes[row, 0].set_xlabel("Predicted Market Value (Million E)", fontsize=11)
    axes[row, 0].set_ylabel("Residual Error (Million E)", fontsize=11)

    sns.histplot(subset['residuals'] / 1e6, kde=True, bins=50, color=hist_colors[label], ax=axes[row, 1])
    axes[row, 1].axvline(x=0, color='black', linestyle='--', linewidth=2)
    axes[row, 1].set_title(f"Error Distribution -- {label} model", fontsize=13, fontweight='bold')
    axes[row, 1].set_xlabel("Residual Error (Million E)", fontsize=11)
    axes[row, 1].set_ylabel("Frequency (Number of Players)", fontsize=11)

plt.suptitle("ScoutAI Diagnostics: Model Health & Variance Analysis (Full vs. Performance-Only)",
             fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../images/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

for label in ['full', 'performance_only']:
    subset = df[df['model_used'] == label]
    print(f"\n--- {label} model residual summary ---")
    print(f"  n              : {len(subset)}")
    print(f"  mean residual  : E{subset['residuals'].mean():,.0f}")
    print(f"  median residual: E{subset['residuals'].median():,.0f}")
    print(f"  std residual   : E{subset['residuals'].std():,.0f}")